In [66]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, Point
import cv2 as cv
from PIL import Image

import sys
import os
from tqdm import tqdm; tqdm.pandas()

from astropy.visualization import make_lupton_rgb
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_adaptive

## Importing Data

In [64]:
data_fold = 'C:/Users/oryan/Documents/mergers_in_desi/MiD-revamped/data'
im_fold = 'E:/GZ-DESI/images-r50-recalib'
save_fold = 'C:/Users/oryan/Documents/mergers_in_desi/MiD-revamped/contour-plots'

In [3]:
data = pd.read_csv(f'{data_fold}/merged-data.csv', index_col = 0)
im_files = glob.glob(f'{im_fold}/*.fits')

## Getting IDs of Downloaded Images

In [4]:
names = []
for i in im_files:
    names.append(os.path.basename(i).replace('-cutout.fits', ''))

## Creating Name Manifest

In [5]:
tmp = zip(names, im_files)

In [6]:
df = pd.DataFrame(tmp).rename(columns = {0 : 'names', 1 : 'file_loc'})

## Calculating the Gini Coefficient

In [7]:
def calc_gini_func(pixels):
    mean_flux = np.mean(abs(pixels))
    ordered_pixels = np.sort(pixels)
    n = len(ordered_pixels)
    
    gini = (((2 * np.arange(1, n + 1)) - n) - 1)*np.abs(ordered_pixels)
        
    normalization =  (mean_flux * n * (n - 1))
    
    return np.sum(gini) / normalization

In [8]:
def conts_to_arr(nested_list):
    contour_arr = np.zeros([len(nested_list),2])
    for i in range(len(nested_list)):
        row = nested_list[i][0]
        contour_arr[i,0] = row[0]
        contour_arr[i,1] = row[1]
    
    return contour_arr

In [9]:
def getting_correct_contours(contours, cen_x, cen_y):
    point = Point(cen_x, cen_y)        
    for i in contours:
        cont_arr = conts_to_arr(i)
        if len(cont_arr) > 2:
            polygon = Polygon(cont_arr)
            if polygon.contains(point):
                return cont_arr
            else:
                continue
        else:
            continue
        
    return 'failed'

In [128]:
def get_galaxy(cutout):
    cutout_int = cutout.copy()
    
    cut = np.percentile(cutout,88)
    cutout_int[cutout_int <= cut] = 0
    cutout_int[cutout_int > cut] = 1
    cutout_int = cutout_int.astype(int)
    
    contours, _ = cv.findContours(cutout_int, cv.RETR_FLOODFILL, cv.CHAIN_APPROX_NONE)
    
    contour_arr = getting_correct_contours(contours, int(cutout.shape[0]/2), int(cutout.shape[1]/2))

    if contour_arr == "failed":
        return 'no-galaxy'
        
    pl = Polygon(contour_arr)
    
    pixels_mask = np.zeros(cutout.shape).astype(bool)
    for i in range(cutout.shape[0]):
        for j in range(cutout.shape[1]):
            pt = Point(i,j)
            if pl.contains(pt):
                pixels_mask[i,j] = True
    pixels_mask = pixels_mask.T
    
    reduced_cutout = cutout[pixels_mask]
    
    xy = pl.exterior.xy
    
    return reduced_cutout

In [129]:
def calc_gini(filepath):
    try:
        data = fits.getdata(filepath)
    except:
        return 'corrupted'
    
    try:
        header = fits.getheader(filepath)
    except:
        return 'corrupted'

    cutout = np.mean(data, axis = 0)

    if np.sum(cutout) == 0:
        return 'empty-image'
    
    reduced_cutout = get_galaxy(cutout)

    if reduced_cutout == 'no-galaxy':
        return 'no-galaxy'
    
    gini = calc_gini_func(reduced_cutout)
#     colour_im = make_lupton_rgb(data[2,:,:], data[1,:,:], data[0,:,:], stretch = 0.1, Q=10)
    
#     im = Image.fromarray(colour_im)
    
#     plt.figure(figsize = (10,10))
#     plt.imshow(im, origin = 'lower')
#     plt.plot(xy[0], xy[1], color = 'red')
#     plt.savefig(f'{save_fold}/{os.path.basename(filepath).replace("-cutout.fits", "")}.jpeg', dpi = 500)
#     plt.close()
        
    return gini

In [130]:
df_ginis = (
    df
    .assign(gini = df.file_loc.progress_apply(lambda x: calc_gini(x)))
)

  0%|          | 0/6675 [00:00<?, ?it/s]C:\Users\oryan\AppData\Local\Continuum\anaconda3\lib\site-packages\ipykernel_launcher.py:13: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  del sys.path[0]
C:\Users\oryan\AppData\Local\Continuum\anaconda3\lib\site-packages\ipykernel_launcher.py:19: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  9%|▊         | 584/6675 [07:51<1:21:57,  1.24it/s] 


KeyboardInterrupt: 